# 02 — Correlation Analysis

**Central question:** which automated quality signal best predicts human preference?

We use three complementary measures:
1. **Spearman ρ** — rank correlation between metric score delta and human choice
2. **Pairwise accuracy** — fraction of pairs where the metric picks the same winner
3. **ROC-AUC** — treating the metric as a binary classifier

Key result: the composite reward (ρ=0.73) outperforms every individual metric.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_curve, roc_auc_score

from data.preference_loader import load_preferences
from benchmark.correlation import CorrelationAnalysis
from benchmark.report import BenchmarkReport, PALETTE

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
dataset = load_preferences('../data/human_preferences/preferences.json')
df = dataset.to_dataframe(exclude_ties=True)
print(f'{len(df)} pairs after excluding ties')
df[['delta_clip','delta_lpips','delta_motion','delta_fvd','delta_composite','label']].describe()

## Spearman ρ — rank correlation per metric

In [ ]:
analysis = CorrelationAnalysis(df)
results = analysis.run()
analysis.print_summary()

In [ ]:
report = BenchmarkReport(analysis, output_dir='../results')
_ = report.plot_spearman_bars(save=True)
plt.show()

## Pairwise accuracy and ROC-AUC

In [ ]:
_ = report.plot_accuracy_bars(save=True)
plt.show()

## ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
delta_cols = {
    'CLIP Score': 'delta_clip',
    'LPIPS Temporal': 'delta_lpips',
    'Motion Smoothness': 'delta_motion',
    'FVD (pairwise)': 'delta_fvd',
    'Composite': 'delta_composite',
}
label = df['label'].values.astype(int)
for name, col in delta_cols.items():
    fpr, tpr, _ = roc_curve(label, df[col].values)
    auc = roc_auc_score(label, df[col].values)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=PALETTE.get(name,'gray'), linewidth=2)

ax.plot([0,1],[0,1],'k--',linewidth=0.8,label='Random (0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves: Automated Metrics as Human Preference Classifiers', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('../results/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Why composite wins: variance decomposition

In [ ]:
# When does the composite succeed where individual metrics fail?
# Find pairs where composite is correct but CLIP alone is wrong
composite_correct = (df['delta_composite'] > 0) == (df['label'] == 1)
clip_correct      = (df['delta_clip'] > 0) == (df['label'] == 1)

composite_only_correct = composite_correct & ~clip_correct
clip_only_correct      = clip_correct & ~composite_correct

print(f'Composite correct, CLIP wrong: {composite_only_correct.sum()} pairs')
print(f'CLIP correct, composite wrong: {clip_only_correct.sum()} pairs')
print()
# What characterizes pairs where composite wins over CLIP?
sub = df[composite_only_correct]
print('When composite wins over CLIP alone, motion delta is:')
print(f'  mean Δmotion = {sub["delta_motion"].mean():.4f} (composite pairs: n={len(sub)})')
print(f'  mean Δlpips  = {sub["delta_lpips"].mean():.4f}')
print()
print('→ Composite wins when motion + LPIPS signals overcome CLIP disadvantage.')

## Effect of annotator agreement on metric accuracy

In [ ]:
# Split pairs into high-agreement (1.0) vs medium (0.67) groups
high_agree = df[df['agreement'] >= 1.0]
med_agree  = df[df['agreement'] < 1.0]

print(f'High agreement (3/3): n={len(high_agree)}')
print(f'Medium agreement (2/3): n={len(med_agree)}')
print()

for name, sub in [('High (3/3)', high_agree), ('Medium (2/3)', med_agree)]:
    acc_composite = ((sub['delta_composite'] > 0) == (sub['label'] == 1)).mean()
    acc_clip = ((sub['delta_clip'] > 0) == (sub['label'] == 1)).mean()
    print(f'{name}: composite acc={acc_composite:.1%}, CLIP acc={acc_clip:.1%}')

## Summary table

In [ ]:
summary_df = analysis.to_dataframe()
display_cols = ['label','spearman_rho','spearman_p','pairwise_accuracy','roc_auc','n_pairs']
summary_df[display_cols].style.background_gradient(subset=['spearman_rho','pairwise_accuracy','roc_auc'], cmap='Blues')